In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Configurer la gestion du bruit avec Estimator

<Accordion>
<AccordionItem title="Versions des packages">

Le code sur cette page a été développé en utilisant les exigences suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Il existe plusieurs façons de gérer le bruit, généralement en utilisant diverses techniques d'atténuation et de suppression d'erreurs pour éviter les erreurs avant qu'elles ne se produisent. Ces techniques entraînent généralement une surcharge de pré-traitement. Par conséquent, il est important d'atteindre un équilibre entre la perfection de tes résultats et la garantie que ton job se termine dans un délai raisonnable.

Estimator prend en charge les techniques de gestion du bruit suivantes. Consulte [Techniques d'atténuation et de suppression d'erreurs](error-mitigation-and-suppression-techniques) pour une explication de chacune. Consulte la section [Paramètres d'erreur personnalisés](#advanced-error) pour les instructions d'activation de ces techniques.

- [découplage dynamique](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options#dynamicaldecouplingoptions)
- [torsion de Pauli](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options)
- [PEA](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options)
- [PEC](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#pec)
- [`resilience_level`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options#resilience_level)
- [TREX](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#measure_mitigation)
- [ZNE](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2#zne)

<span id="resilience"></span>
## Niveau de résilience
Le `resilience_level` spécifie la résilience à construire contre
les erreurs. Des niveaux plus élevés génèrent des résultats plus précis, au prix de
temps de traitement plus longs. Les niveaux de résilience peuvent être utilisés pour configurer le
compromis coût/précision lors de l'application de la gestion du bruit à ta
requête de primitive. La gestion du bruit réduit les erreurs (biais) dans les résultats en traitant
les sorties d'une collection, ou d'un ensemble, de circuits apparentés. Le
degré de réduction des erreurs dépend de la méthode appliquée. Le niveau de résilience
abstrait le choix détaillé de la méthode de gestion du bruit pour permettre
aux utilisateurs de raisonner sur le compromis coût/précision approprié à
leur application.

Compte tenu de cela, chaque niveau correspond à une méthode ou à des méthodes avec
un niveau croissant de surcharge d'échantillonnage quantique pour te permettre d'expérimenter
avec différents compromis temps-précision. Le tableau suivant te montre
quels niveaux et méthodes correspondantes sont disponibles pour chacune des
primitives.

<span id="resilience-table"></span>

| Niveau de résilience | Description                                                                                            | Technique                                                                 |
|------------------|-------------------------------------------------------------------------------------------------------|---------------------------------------------------------------------------|
| 0                | Aucune atténuation                                                                                        | Aucune                                                                      |
| 1 [Par défaut]      | Coûts d'atténuation minimaux : Atténuer les erreurs associées aux erreurs de lecture                               | Torsion de mesure [Twirled Readout Error eXtinction (TREX)](/guides/error-mitigation-and-suppression-techniques#trex)             |
| 2                | Coûts d'atténuation moyens. Réduit généralement le biais dans les estimateurs, mais n'est pas garanti d'être sans biais. | Niveau 1 + [Extrapolation zéro-bruit (ZNE)](/guides/error-mitigation-and-suppression-techniques#zne) et torsion de gate

> **Info:** Les niveaux de résilience sont actuellement en version bêta, donc la surcharge d'échantillonnage et
> la qualité de la solution varieront d'un circuit à l'autre. De nouvelles fonctionnalités,
> des options avancées et des outils de gestion seront publiés en continu. Les méthodes spécifiques de gestion du bruit ne sont pas garanties d'être
> appliquées à chaque niveau de résilience.

### Configurer Estimator avec des niveaux de résilience
Tu peux utiliser des niveaux de résilience pour spécifier des techniques de gestion du bruit, ou tu peux définir des techniques personnalisées individuellement comme décrit dans [Paramètres d'erreur personnalisés](#advanced-error).

> **Note:** Toutes les options que tu spécifies manuellement en plus du niveau de résilience sont appliquées en plus de l'ensemble d'options de base définies par le niveau de résilience. Par conséquent, en principe, tu pourrais définir le niveau de résilience à 1, mais ensuite désactiver l'atténuation de mesure, bien que cela ne soit pas conseillé.
> 
> Par exemple, définir le niveau de résilience à 0 désactive `zne_mitigation`, mais `estimator.options.resilience.zne_mitigation = True` remplace cette valeur.

### Exemple
Le code suivant active ZNE, TREX et la torsion de gate en
définissant `resilience_level 2`.

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(backend, options={"resilience_level": 2})

<span id="advanced-error"></span>
## Paramètres de gestion du bruit personnalisés
Tu peux activer et désactiver individuellement les méthodes de gestion du bruit en utilisant les [options Estimator](/guides/estimator-options).

> **Note:** Toutes les options ne fonctionnent pas ensemble sur tous les types de circuits. Consulte la [table de compatibilité des fonctionnalités](estimator-options#options-compatibility-table) pour plus de détails.

### Exemple

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(backend)
options = estimator.options
# Turn on gate twirling.
options.twirling.enable_gates = True
# Turn on measurement error mitigation.
options.resilience.measure_mitigation = True

print(
    f">>> gate twirling is turned on: {estimator.options.twirling.enable_gates}"
)
print(
    f">>> measurement error mitigation is turned on: {estimator.options.resilience.measure_mitigation}"
)

>>> gate twirling is turned on: True
>>> measurement error mitigation is turned on: True


## Turn off all error mitigation

For instructions to turn off all error mitigation see the [Turn off all error suppression and mitigation](estimator-options#no-error-mitigation) section in the Estimator options guide.

## Next steps

<Admonition type="tip" title="Recommendations">
  - Walk through an example that uses error mitigation in the [Cost function lesson](/learning/courses/variational-algorithm-design/cost-functions) in IBM Quantum Learning.
  - Learn more about [error mitigation and error suppression techniques](error-mitigation-and-suppression-techniques).
  - Explore [Estimator options](/docs/guides/estimator-options).
  - Decide what [execution mode](/docs/guides/execution-modes) to run your job in.
</Admonition>